# 4GB VRAM verification on Colab (NVIDIA T4)

Runs the same code as on the development machine, on a CUDA GPU, and records **real VRAM usage via NVML**:
- `scripts/measure_memory.py`: llama.cpp buffer accounting per device + NVML device/process VRAM, peak sampled every 20 ms during full answers
- `benchmark.py`: TTFT / decode TPS / moderation overhead, moderation off vs on

A T4 has 16 GB and cannot be hard-capped at 4 GB, so the check is: **peak NVML VRAM (whole device, including the CUDA context) < 4096 MiB**.

Runtime → Change runtime type → **T4 GPU**.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used,driver_version --format=csv

In [ ]:
# uv + repo
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] = "/root/.local/bin:" + os.environ["PATH"]
REPO = "https://github.com/popo9790/Laptop-AI-asistant.git"  # adjust if the repo URL differs
!rm -rf /content/repo && git clone -q $REPO /content/repo
%cd /content/repo

In [ ]:
# llama-cpp-python with CUDA (source build, ~10-15 min on Colab)
os.environ["CMAKE_ARGS"] = "-DGGML_CUDA=on"
os.environ["FORCE_CMAKE"] = "1"
!uv sync 2>&1 | tail -3
!uv run python -c "import llama_cpp; print(llama_cpp.__version__, llama_cpp.llama_supports_gpu_offload())"

In [ ]:
# models + data + index (data/processed/ is generated, not committed). Stops on the first error.
!set -e; uv run python scripts/download_models.py && uv run python scripts/prepare_data.py && uv run python scripts/build_index.py
!ls -la data/processed/

In [ ]:
# VRAM: default config, then the reduced compute buffer (full output, errors not hidden)
FILTER = "ggml_|llama_model_loader|print_info|load_tensors: +layer|llama_kv_cache: layer|llama_memory_recurrent, layer"
!uv run python scripts/measure_memory.py --out logs/memory_colab_ub512.json 2>&1 | grep -v -E "$FILTER"
!uv run python scripts/measure_memory.py --n-ubatch 128 --out logs/memory_colab_ub128.json 2>&1 | grep -v -E "$FILTER"

In [ ]:
# performance with / without moderation (interleaved)
!uv run python benchmark.py --runs 3 --out logs/benchmark_colab.json 2>&1 | grep -v -E "ggml_|llama_context"

In [ ]:
import json
for f in ["logs/memory_colab_ub512.json", "logs/memory_colab_ub128.json"]:
    r = json.load(open(f))
    peak = r["peak_during_requests"]["vram_device_mb"]
    print(f, "| accelerator buffers", r["accelerator_total_mib"], "MiB | peak NVML device VRAM", peak,
          "MB |", "PASS" if peak and peak < 4096 else "CHECK", "(< 4096 MiB)")

In [ ]:
# optional: tests and moderation eval on GPU
!uv run pytest -q
!uv run python eval_moderation.py --input-only --out logs/eval_moderation_colab_input.json | tail -20